# AIRL Mode Control Training

Train the four publication configurations in one place:

- State only
- State + comfort (zonal)
- State + energy (zonal)
- State + energy + comfort (zonal)

This notebook shares data preparation, expert trajectory extraction, and the dynamics model, then trains each reward variant sequentially and writes checkpoints to `notebooks/outputs/`.


In [ ]:
import os
import sys
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def _iter_project_candidates(start: Path):
    seen = set()

    first_pass = [start]
    try:
        first_pass.extend(child for child in start.iterdir() if child.is_dir())
    except OSError:
        pass

    for candidate in first_pass:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate

    for candidate in start.parents:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        yield candidate


def _locate_project_root(start: Path) -> Path:
    candidates = []
    for candidate in _iter_project_candidates(start):
        try:
            has_layout = (candidate / 'src').is_dir() and (candidate / 'configs').is_dir()
        except OSError:
            has_layout = False
        if not has_layout:
            continue

        score = 0
        if (candidate / 'notebooks' / 'README.md').is_file():
            score += 2
        if (candidate / 'tests' / 'test_standalone_repo.py').is_file():
            score += 2
        candidates.append((score, candidate))

    if not candidates:
        raise RuntimeError('Unable to locate project root from ' + str(start))

    best_score, best_candidate = candidates[0]
    for score, candidate in candidates[1:]:
        if score > best_score:
            best_score, best_candidate = score, candidate
    return best_candidate


try:
    NOTEBOOK_PATH = Path(__file__).resolve()
    NOTEBOOK_DIR = NOTEBOOK_PATH.parent
except NameError:
    NOTEBOOK_PATH = None
    NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = _locate_project_root(NOTEBOOK_DIR)
if NOTEBOOK_DIR.name != 'notebooks' and (PROJECT_ROOT / 'notebooks').is_dir():
    NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

plt.style.use('seaborn-v0_8')
print(f'Notebook path: {NOTEBOOK_PATH}')
print(f'Notebook directory: {NOTEBOOK_DIR}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Using torch {torch.__version__}')

from configs.training_config import get_full_training_config
from data_processing import (
    extract_expert_trajectories,
    inverse_normalize,
    load_and_filter_data,
    set_random_seed,
    setup_scalers,
    split_train_val,
)
from models import load_dynamics_model
from airl_environment import create_airl_environment
from airl_training import collect_trajectories, flatten_action, run_airl_training


## Configure Experiment

The common setup below is shared across all four reward configurations. Set `RUN_VARIANT_KEYS` if you want to rerun only a subset, and set `ACTIVE_VARIANT_KEY` to choose which trained variant the later qualitative plots should inspect.


In [ ]:
BASE_CONFIG = get_full_training_config()
BASE_CONFIG.data.data_file = 'l14_merged_data_with_rain.csv'
BASE_CONFIG.data.min_trajectory_length = 690

BASE_CONFIG.training.n_airl_iters = 20
BASE_CONFIG.training.batch_days = 16
BASE_CONFIG.training.val_days = 11
BASE_CONFIG.training.ppo_updates = 10
BASE_CONFIG.training.ppo_epochs = 6
BASE_CONFIG.training.policy_lr = 5e-4
BASE_CONFIG.training.reward_lr = 5e-4
BASE_CONFIG.training.reward_update_steps = 5

INITIAL_REWARD_UPDATE_STEPS = 5
INITIAL_REWARD_MIN_MARGIN = 0.02
INITIAL_REWARD_MAX_ATTEMPTS = 6

BASE_CONFIG.environment.comfort_max = 27.0
REPORT_COMFORT_CEILING = 30.0
DATES_TO_EXCLUDE = {(7, 17), (8, 9)}

VARIANT_SPECS = [
    {
        'key': 'state_only',
        'label': 'State Only',
        'reward_type': 'temps_hold_dwell_prev_time_gru',
        'color': plt.get_cmap('Set2')(0),
    },
    {
        'key': 'comfort_zonal',
        'label': 'State + Comfort',
        'reward_type': 'temps_hold_dwell_prev_time_comfort_zonal_gru',
        'color': plt.get_cmap('Set2')(1),
    },
    {
        'key': 'energy_zonal',
        'label': 'State + Energy',
        'reward_type': 'temps_hold_dwell_prev_time_energy_zonal_gru',
        'color': plt.get_cmap('Set2')(2),
    },
    {
        'key': 'energy_comfort_zonal',
        'label': 'State + Energy + Comfort',
        'reward_type': 'temps_hold_dwell_prev_time_energy_comfort_zonal_gru',
        'color': plt.get_cmap('Set2')(3),
    },
]

RUN_VARIANT_KEYS = None
ACTIVE_VARIANT_KEY = 'state_only'

set_random_seed(BASE_CONFIG.data.random_seed)
device = torch.device(BASE_CONFIG.training.device)

selected_key_set = set(RUN_VARIANT_KEYS) if RUN_VARIANT_KEYS else {spec['key'] for spec in VARIANT_SPECS}
selected_specs = [spec for spec in VARIANT_SPECS if spec['key'] in selected_key_set]
if not selected_specs:
    raise ValueError('RUN_VARIANT_KEYS did not match any known variant keys.')
if ACTIVE_VARIANT_KEY not in {spec['key'] for spec in selected_specs}:
    ACTIVE_VARIANT_KEY = selected_specs[0]['key']

variant_table = pd.DataFrame([
    {
        'key': spec['key'],
        'label': spec['label'],
        'reward_type': spec['reward_type'],
    }
    for spec in selected_specs
])
display(variant_table)

print(f'Device: {device}')
print(f'Active variant for later plots: {ACTIVE_VARIANT_KEY}')
print(BASE_CONFIG)


## Shared Data Preparation

These steps run once and are reused across every reward configuration.


In [ ]:
data_path = PROJECT_ROOT / BASE_CONFIG.data.data_file
print(f'Loading data from {data_path}')

daily_dataframes = load_and_filter_data(
    file_path=str(data_path),
    min_length=BASE_CONFIG.data.min_trajectory_length,
)
print(f'Total usable days before exclusions: {len(daily_dataframes)}')

filtered_daily = []
removed_labels = []
for df in daily_dataframes:
    day = pd.to_datetime(df['date'].iloc[0])
    if (day.month, day.day) in DATES_TO_EXCLUDE:
        removed_labels.append(day.strftime('%Y-%m-%d'))
        continue
    filtered_daily.append(df)

daily_dataframes = filtered_daily
print(f'Removed {len(removed_labels)} days: {removed_labels}')
print(f'Remaining days: {len(daily_dataframes)}')

train_dfs, val_dfs = split_train_val(
    daily_dataframes,
    split_ratio=BASE_CONFIG.data.train_val_split,
    shuffle=True,
    random_state=0,
)
print(f'Train days: {len(train_dfs)} | Validation days: {len(val_dfs)}')

scalers = setup_scalers(train_dfs, val_dfs)
print(f'Created scalers for {len(scalers)} features')

expert_trajs = extract_expert_trajectories(
    train_dfs,
    scalers,
    window_col='Z1 Windows Open Close Status',
    comfort_max=REPORT_COMFORT_CEILING,
    comfort_min=BASE_CONFIG.environment.comfort_min,
)
print(f'Expert training trajectories: {len(expert_trajs)}')
print(f"Total expert timesteps: {sum(len(traj['states']) for traj in expert_trajs)}")

expert_trajs_val = extract_expert_trajectories(
    val_dfs,
    scalers,
    window_col='Z1 Windows Open Close Status',
    comfort_max=REPORT_COMFORT_CEILING,
    comfort_min=BASE_CONFIG.environment.comfort_min,
)
print(f'Expert validation trajectories: {len(expert_trajs_val)}')


In [ ]:
model_path = PROJECT_ROOT / BASE_CONFIG.dynamics_model_path
print(f'Loading dynamics model from {model_path}')

dynamics_model = load_dynamics_model(
    model_path=str(model_path),
    device=torch.device('cpu'),
    hidden_dim=BASE_CONFIG.model.dynamics_hidden_dim,
    output_dim=BASE_CONFIG.model.dynamics_output_dim,
    ac_dim=BASE_CONFIG.model.dynamics_ac_channels,
    nv_dim=BASE_CONFIG.model.dynamics_nv_channels,
)
dynamics_model.eval()


In [ ]:
def build_variant_config(spec: Dict[str, Any]):
    config = deepcopy(BASE_CONFIG)
    config.model.reward_type = spec['reward_type']
    return config


def build_env_bundle(config):
    train_env = create_airl_environment(
        daily_data_list=train_dfs,
        dynamics_model=dynamics_model,
        scalers=scalers,
        num_zones=config.model.num_zones,
        comfort_min=config.environment.comfort_min,
        comfort_max=config.environment.comfort_max,
        heavy_wind_threshold=config.environment.heavy_wind_threshold,
        policy_time_features=True,
    )
    val_env = create_airl_environment(
        daily_data_list=val_dfs,
        dynamics_model=dynamics_model,
        scalers=scalers,
        num_zones=config.model.num_zones,
        comfort_min=config.environment.comfort_min,
        comfort_max=config.environment.comfort_max,
        heavy_wind_threshold=config.environment.heavy_wind_threshold,
        policy_time_features=True,
    )
    report_env_kwargs = dict(
        daily_data_list=val_dfs,
        dynamics_model=dynamics_model,
        scalers=scalers,
        num_zones=config.model.num_zones,
        comfort_min=config.environment.comfort_min,
        comfort_max=REPORT_COMFORT_CEILING,
        heavy_wind_threshold=config.environment.heavy_wind_threshold,
        policy_time_features=True,
    )

    initial_state = train_env.reset(day_index=0)
    dummy_action = {
        'change': 0,
        'local_cooling': train_env.prev_lc_temps.copy(),
        'supply_temps': train_env.prev_supply_temps[:config.model.num_zones].copy(),
    }
    action_dim = flatten_action(dummy_action).shape[0]

    return {
        'train_env': train_env,
        'val_env': val_env,
        'report_env_kwargs': report_env_kwargs,
        'state_dim': initial_state.shape[0],
        'action_dim': action_dim,
    }


def save_final_artifacts(config, policy, reward_fn):
    output_dir = PROJECT_ROOT / config.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    reward_tag = config.model.reward_type
    paths = {
        'policy_final': output_dir / f'airl_mode_policy_{reward_tag}_final.pth',
        'reward_final': output_dir / f'airl_mode_reward_{reward_tag}_final.pth',
        'policy_final_state': output_dir / f'airl_mode_policy_{reward_tag}_final_state.pth',
        'reward_final_state': output_dir / f'airl_mode_reward_{reward_tag}_final_state.pth',
    }
    torch.save(policy.state_dict(), paths['policy_final'])
    torch.save(reward_fn.state_dict(), paths['reward_final'])
    torch.save(policy.state_dict(), paths['policy_final_state'])
    torch.save(reward_fn.state_dict(), paths['reward_final_state'])
    return paths


def summarize_eval_trajs(eval_trajs):
    rows = []
    for idx, traj in enumerate(eval_trajs):
        energy_kwh = float(np.sum(traj.get('energy_features', [])))
        comfort_deg_c_min = float(np.sum(traj.get('comfort_features', [])))
        window_switches = int(np.sum(np.asarray(traj.get('actions', []), dtype=np.float32)[:, 0])) if len(traj.get('actions', [])) else 0
        rows.append({
            'day_index': idx,
            'energy_kwh': energy_kwh,
            'comfort_degC_min': comfort_deg_c_min,
            'window_switches': window_switches,
            'episode_length': len(traj.get('states', [])),
        })

    metrics_df = pd.DataFrame(rows)
    if metrics_df.empty:
        return metrics_df, {}

    summary = {
        'val_energy_kwh_mean': float(metrics_df['energy_kwh'].mean()),
        'val_comfort_degC_min_mean': float(metrics_df['comfort_degC_min'].mean()),
        'val_window_switches_mean': float(metrics_df['window_switches'].mean()),
    }
    return metrics_df, summary


def train_variant(spec: Dict[str, Any]):
    config = build_variant_config(spec)
    env_bundle = build_env_bundle(config)

    training_kwargs = dict(
        env=env_bundle['train_env'],
        validation_env=env_bundle['val_env'],
        expert_trajs=expert_trajs,
        state_dim=env_bundle['state_dim'],
        action_dim=env_bundle['action_dim'],
        device=device,
        n_iters=config.training.n_airl_iters,
        policy_lr=config.training.policy_lr,
        reward_lr=config.training.reward_lr,
        reward_update_steps=config.training.reward_update_steps,
        ppo_updates=config.training.ppo_updates,
        reward_type=config.model.reward_type,
        policy_weight_decay=1e-5,
        grad_clip_max_norm=2.0,
        entropy_coeff=0.005,
        initial_reward_update_steps=INITIAL_REWARD_UPDATE_STEPS,
        initial_reward_min_margin=INITIAL_REWARD_MIN_MARGIN,
        initial_reward_max_attempts=INITIAL_REWARD_MAX_ATTEMPTS,
        collect_days=config.training.batch_days,
        ppo_batch_days=config.training.batch_days,
        ppo_epochs=config.training.ppo_epochs,
        validation_days=config.training.val_days,
        output_dir=config.output_dir,
    )

    trained_policy, learned_reward = run_airl_training(**training_kwargs)
    checkpoint_paths = save_final_artifacts(config, trained_policy, learned_reward)

    training_metrics = pd.DataFrame(getattr(trained_policy, 'training_metrics', []))
    if not training_metrics.empty:
        training_metrics.index = np.arange(1, len(training_metrics) + 1)
        training_metrics.index.name = 'Iteration'

    eval_env = create_airl_environment(**env_bundle['report_env_kwargs'])
    eval_trajs = collect_trajectories(
        eval_env,
        trained_policy,
        day_indices=range(len(val_dfs)),
    )
    validation_metrics, validation_summary = summarize_eval_trajs(eval_trajs)

    return {
        'spec': spec,
        'config': config,
        'policy': trained_policy,
        'reward_fn': learned_reward,
        'training_metrics': training_metrics,
        'expert_metrics': getattr(trained_policy, 'expert_metrics', {}),
        'validation_metrics': validation_metrics,
        'validation_summary': validation_summary,
        'eval_trajs': eval_trajs,
        'report_env_kwargs': env_bundle['report_env_kwargs'],
        'state_dim': env_bundle['state_dim'],
        'action_dim': env_bundle['action_dim'],
        'checkpoint_paths': checkpoint_paths,
    }


def plot_variant_validation_control(run: Dict[str, Any], num_days: int = 11):
    replay_env = create_airl_environment(**run['report_env_kwargs'])
    config = run['config']
    policy = run['policy']
    comfort_limit = REPORT_COMFORT_CEILING
    outdoor_col = 'OutdoorTemperatureWindow'

    num_days = min(num_days, len(val_dfs))
    fig, axes = plt.subplots(num_days, 1, figsize=(14, 3.5 * num_days), sharex=True)
    if num_days == 1:
        axes = [axes]

    for row_idx in range(num_days):
        ax = axes[row_idx]
        state = replay_env.reset(day_index=row_idx)
        avg_zone_trace = []
        outdoor_trace = []
        window_trace = []

        done = False
        while not done:
            action, _ = policy.get_action(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
            state, _, done, info = replay_env.step(action)

            zone_c = [
                inverse_normalize(replay_env.zone_temps[i], replay_env.columns['zone_cols'][i], replay_env.scalers)
                for i in range(config.model.num_zones)
            ]
            avg_zone_trace.append(float(np.mean(zone_c)))

            outside_val = info.get('outdoor_temp_c')
            if outside_val is None:
                outside_idx = len(replay_env.zone_temps)
                if state.shape[0] > outside_idx:
                    outside_val = inverse_normalize(state[outside_idx], outdoor_col, replay_env.scalers)
            outdoor_trace.append(float(outside_val) if outside_val is not None else np.nan)
            window_trace.append(int(info.get('window', replay_env.window_prev_state)))

        minutes = np.arange(len(avg_zone_trace))
        ax.plot(minutes, outdoor_trace, color='tab:orange', linewidth=1.6, zorder=2.5)
        ax.axhline(comfort_limit, color='tab:red', linestyle='--', linewidth=1.2, zorder=2)
        ax.plot(minutes, avg_zone_trace, color='tab:blue', linewidth=2.0, zorder=3)

        window_array = np.array(window_trace, dtype=int)
        change_points = np.where(np.diff(window_array) != 0)[0] + 1
        segment_starts = np.concatenate(([0], change_points))
        segment_ends = np.concatenate((change_points, [len(window_array)]))

        for start, end in zip(segment_starts, segment_ends):
            status = window_array[start]
            shade_color = '#d9d9d9' if status == 0 else '#8fb8de'
            shade_alpha = 0.18 if status == 0 else 0.35
            ax.axvspan(start, end, color=shade_color, alpha=shade_alpha, zorder=0)

        ax.set_title(f"{run['spec']['label']} | validation day {row_idx}")
        ax.set_ylabel('Temperature (°C)')
        ax.grid(True, linestyle='--', alpha=0.3)

    legend_handles = [
        Line2D([0], [0], color='tab:blue', linewidth=2.0, label='Avg zone temp (°C)'),
        Line2D([0], [0], color='tab:orange', linewidth=1.6, label='Outdoor temp (°C)'),
        Line2D([0], [0], color='tab:red', linestyle='--', linewidth=1.2, label=f'Comfort limit ({comfort_limit:.0f}°C)'),
        Patch(facecolor='#d9d9d9', edgecolor='none', alpha=0.18, label='Window closed'),
        Patch(facecolor='#8fb8de', edgecolor='none', alpha=0.35, label='Window open'),
    ]
    axes[0].legend(handles=legend_handles, loc='upper right', frameon=False)
    axes[-1].set_xlabel('Minute index')
    fig.tight_layout()
    plt.show()


## Train AIRL Policies

This cell trains each selected reward variant in sequence, saves its final checkpoints, and collects a compact validation summary.


In [ ]:
variant_runs: Dict[str, Dict[str, Any]] = {}
summary_rows: List[Dict[str, Any]] = []

for spec in selected_specs:
    print()
    print('=' * 80)
    print(f"Training {spec['label']} ({spec['reward_type']})")
    print('=' * 80)
    run = train_variant(spec)
    variant_runs[spec['key']] = run

    training_metrics = run['training_metrics']
    validation_summary = run['validation_summary']
    expert_metrics = run['expert_metrics'] if isinstance(run['expert_metrics'], dict) else {}
    final_train = training_metrics.iloc[-1].to_dict() if not training_metrics.empty else {}

    summary_rows.append({
        'key': spec['key'],
        'label': spec['label'],
        'reward_type': spec['reward_type'],
        'final_train_energy_per_hour': final_train.get('avg_energy_per_hour'),
        'final_train_comfort_pct': final_train.get('avg_comfort_pct'),
        'final_train_window_switches': final_train.get('avg_window_switches'),
        'expert_energy_per_hour': expert_metrics.get('avg_energy_per_hour'),
        'expert_comfort_pct': expert_metrics.get('avg_comfort_pct'),
        'expert_window_switches': expert_metrics.get('avg_window_switches'),
        'val_energy_kwh_mean': validation_summary.get('val_energy_kwh_mean'),
        'val_comfort_degC_min_mean': validation_summary.get('val_comfort_degC_min_mean'),
        'val_window_switches_mean': validation_summary.get('val_window_switches_mean'),
        'policy_final_state_path': str(run['checkpoint_paths']['policy_final_state'].relative_to(PROJECT_ROOT)),
        'reward_final_state_path': str(run['checkpoint_paths']['reward_final_state'].relative_to(PROJECT_ROOT)),
    })

run_summary_df = pd.DataFrame(summary_rows)
display(run_summary_df)


## Compare Variants

Review the compact validation summary and how each configuration evolved over training.


In [ ]:
if 'run_summary_df' not in globals() or run_summary_df.empty:
    print('No variant runs available; execute the training cell first.')
else:
    summary_view = run_summary_df[[
        'label',
        'reward_type',
        'val_energy_kwh_mean',
        'val_comfort_degC_min_mean',
        'val_window_switches_mean',
        'policy_final_state_path',
        'reward_final_state_path',
    ]].copy()
    display(summary_view)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = [
        ('val_energy_kwh_mean', 'Validation energy (kWh/day)'),
        ('val_comfort_degC_min_mean', 'Validation comfort violation'),
        ('val_window_switches_mean', 'Validation window switches/day'),
    ]
    for ax, (metric_key, title) in zip(axes, metrics):
        ax.bar(
            run_summary_df['label'],
            run_summary_df[metric_key],
            color=[variant_runs[key]['spec']['color'] for key in run_summary_df['key']],
        )
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=25)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
if not variant_runs:
    print('No variant runs available; execute the training cell first.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)
    metric_specs = [
        ('avg_energy_per_hour', 'Energy per hour (kWh/h)'),
        ('avg_comfort_pct', 'Comfort violation (%)'),
        ('avg_window_switches', 'Window switches/day'),
    ]
    for spec in selected_specs:
        run = variant_runs[spec['key']]
        history = run['training_metrics']
        if history.empty:
            continue
        x_vals = history.index.to_numpy(dtype=int)
        for ax, (metric_key, title) in zip(axes, metric_specs):
            ax.plot(x_vals, history[metric_key], marker='o', label=spec['label'], color=spec['color'])
            ax.set_title(title)
            ax.set_xlabel('AIRL iteration')
            ax.grid(True, linestyle='--', alpha=0.3)
    axes[0].set_ylabel('Metric value')
    axes[-1].legend(frameon=False, loc='best')
    fig.tight_layout()
    plt.show()


## Inspect One Variant

Set `ACTIVE_VARIANT_KEY` in the config cell to choose which trained policy the remaining qualitative plots should use.


In [ ]:
if ACTIVE_VARIANT_KEY not in variant_runs:
    raise KeyError(f'Active variant {ACTIVE_VARIANT_KEY!r} is not available in variant_runs.')

active_run = variant_runs[ACTIVE_VARIANT_KEY]
print(f"Active variant: {active_run['spec']['label']} ({active_run['spec']['reward_type']})")
display(active_run['training_metrics'])
display(active_run['validation_metrics'])
pd.Series({k: str(v.relative_to(PROJECT_ROOT)) for k, v in active_run['checkpoint_paths'].items()}, name='checkpoint_path').to_frame()


In [ ]:
plot_variant_validation_control(active_run, num_days=min(11, len(val_dfs)))


For deeper side-by-side qualitative comparison across all four trained policies, use `airl_four_config_daily_comparison_time_zonal.ipynb` after this notebook has generated the checkpoints.
